# Mechanistic Debugging of Agentic Sabotage (`Gemma-2-9B-IT` + `Gemma Scope 16k`)

Reproducible notebook for running `Gemma-2-9B-IT` (`bfloat16`) activation caching, sparse autoencoder circuit isolation, and multi-layer steering evaluation on a 24 GB+ CUDA GPU.

### Workflow
1. Select an **L4** or **A100** GPU runtime (`Runtime -> Change runtime type`).
2. Run all cells (`Runtime -> Run all`). Cell 2 automatically clones the repository and installs the package.

In [ ]:
!nvidia-smi -L
!pip install -q transformers accelerate huggingface-hub safetensors scikit-learn seaborn python-dotenv rich

In [ ]:
import os
from pathlib import Path

try:
    from google.colab import userdata
    tok = userdata.get('HF_TOKEN')
    if tok:
        os.environ['HF_TOKEN'] = tok
except Exception:
    pass

if not Path('/content/agentic-sabotage').exists():
    !git clone https://github.com/danielgaskins/agentic-sabotage.git /content/agentic-sabotage

%cd /content/agentic-sabotage
!pip install -q --no-deps -e .

In [ ]:
!python -m agentic_sabotage.cli cache-activations --model gemma-2-9b-it --limit 400 --max-empirical-rollouts 100

In [ ]:
!python -m agentic_sabotage.cli analyze-circuits --model gemma-2-9b-it

In [ ]:
!python -m agentic_sabotage.cli run-steering --model gemma-2-9b-it --n-eval-pressured 48 --n-eval-controls 30

In [ ]:
!python -m agentic_sabotage.cli plot-all --model gemma-2-9b-it

import tarfile
from pathlib import Path
from IPython.display import Image, display

for png in sorted(Path('figures/gemma-2-9b-it').glob('*.png')):
    display(Image(filename=str(png), width=780))

with tarfile.open('/content/gemma2_9b_results.tar.gz', 'w:gz') as out_tar:
    for f in Path('results/gemma-2-9b-it').glob('*.json'):
        out_tar.add(f)
    if Path('results/gemma-2-9b-it/steering_directions.npz').exists():
        out_tar.add('results/gemma-2-9b-it/steering_directions.npz')
    for f in Path('figures/gemma-2-9b-it').glob('*.png'):
        out_tar.add(f)